## cNF vs Bayesian regression: distributional comparison (per-ADM1 EMD)

Reviewer point 3: the paper compares cNF and Bayesian regression on aggregate sub-national means (Fig 6A) and pivots to 'cNF gives full distributions' as the differentiator, but never quantifies that. Here we compute per-ADM1 EMD between the held-out truth and (a) the cNF-generated samples, (b) one draw per household from the Bayesian posterior predictive distribution, for each (dataset, seed, ADM1, target variable).

The Bayesian baseline is multivariate linear regression with flat prior. Its posterior predictive for household $i$ is $\mathcal{N}(\hat\mu_i,\,\hat\sigma_i^2)$, with parameters already saved per-household (in raw units, after `inverse_transform`) as `bayes_mu.csv` / `bayes_std.csv` for each seed. Drawing one value per household gives a synthetic distribution per ADM1 directly comparable to cNF.

All three streams (truth, cNF, Bayesian) are aligned by row order to `df_full = full.csv`. The cNF outputs come from the Bayesian-experiment train.py (`rnvp_generated_run{run_idx}.csv`), which is in the SAME raw units as `bayes_mu.csv` and `full.csv`. Same noise-floor protocol as the main pipeline.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, wilcoxon

EXP_BASE   = "/data/shared/fsibilla/clean_code/Q1/experiments"
BAYES_BASE = "/data/shared/fsibilla/clean_code/Q1/bayesian"
OUT_DIR    = "/data/shared/fsibilla/clean_code/Q1/across_experiments_eval"

SEEDS         = [1, 2, 3, 4, 5]
RUN_IDX       = 0     # which RealNVP run to use as the cNF realisation per seed (0..4 available)
BINS          = 200
N_SUBSAMPLES  = 20

DATASETS = {
    "eth_micron": {"adm1": "adm1name",
                   "targets": ["va_ai","fol_ai","vb12_ai","fe_ai","zn_ai","avg_adult_education","log_exp"]},
    "lka_micron": {"adm1": "adm1name",
                   "targets": ["vita_rae_mcg","folate_mcg","vitb12_mcg","fe_mg","zn_mg","avg_adult_education","log_exp"]},
    "lka_vam":    {"adm1": "adm1name",
                   "targets": ["education_score","log_income","space_per_person","FES","FCS","rCSI"]},
    "moz_vam":    {"adm1": "adm1name",
                   "targets": ["FCS","rCSI","FGVitA","FGProtein","FGHIron"]},
    "nga_micron": {"adm1": "adm1name",
                   "targets": ["va_ai","fol_ai","vb12_ai","fe_ai","zn_ai","avg_adult_education","log_exp"]},
    "nga_mics":   {"adm1": "adm1name",
                   "targets": ["space_per_person","avg_adult_education","wscore"]},
    "yem_mvam":   {"adm1": "adm1name",
                   "targets": ["log_exp_pp","rCSI","FCS"]},
    "zwe_mics":   {"adm1": "adm1name",
                   "targets": ["space_per_person","avg_adult_education","wscore"]},
}

In [ ]:
def _finite_1d(x):
    x = np.asarray(x, dtype=float).ravel()
    return x[np.isfinite(x)]

def emd_1d_density(x, y, bins=200, value_range=None):
    x = _finite_1d(x); y = _finite_1d(y)
    if len(x) == 0 or len(y) == 0: return np.nan
    if value_range is None:
        z = np.concatenate([x, y])
        lo, hi = float(np.min(z)), float(np.max(z))
        if hi <= lo: return 0.0
        value_range = (lo, hi)
    hx, edges = np.histogram(x, bins=bins, range=value_range, density=True)
    hy, _     = np.histogram(y, bins=edges, density=True)
    dx = np.diff(edges)
    return float(np.sum(np.abs(np.cumsum(hx * dx) - np.cumsum(hy * dx)) * dx))

def emd_1d_match(x_ref, x_cand, n_subsamples=20, rng=None, bins=200, value_range=None):
    """Same protocol as emd_density_1d_match in the main pipeline."""
    xr = _finite_1d(x_ref); xc = _finite_1d(x_cand)
    if len(xr) == 0 or len(xc) == 0: return np.nan
    n = len(xr)
    if len(xc) <= n:
        return emd_1d_density(xr, xc, bins=bins, value_range=value_range)
    if rng is None: rng = np.random.default_rng(0)
    vals = []
    for _ in range(n_subsamples):
        idx = rng.choice(len(xc), size=n, replace=False)
        vals.append(emd_1d_density(xr, xc[idx], bins=bins, value_range=value_range))
    return float(np.mean(vals))

In [ ]:
def load_truth(dataset, adm1_col, targets):
    """Truth in raw units, with adm1 column."""
    full = pd.read_csv(os.path.join(EXP_BASE, dataset, "full.csv"))
    keep = [adm1_col] + [c for c in targets if c in full.columns]
    full = full[keep].dropna(subset=keep).reset_index(drop=True)
    return full

def load_seed_outputs(dataset, seed, run_idx, adm1_col, targets, full):
    """Load Bayesian mu/std and one cNF run for one seed.
    Bayesian outputs are aligned by row order to full.csv (one row per
    full-data household). Attach adm1 by index to allow per-region grouping.
    """
    seed_dir = os.path.join(BAYES_BASE, dataset, "results", f"seed_{seed}")
    mu  = pd.read_csv(os.path.join(seed_dir, "bayes_mu.csv"))
    std = pd.read_csv(os.path.join(seed_dir, "bayes_std.csv"))
    cnf = pd.read_csv(os.path.join(seed_dir, f"rnvp_generated_run{run_idx}.csv"))
    # Sanity: lengths must match. (full.csv was the basis for Bayesian/cNF generation.)
    n_full_raw = len(pd.read_csv(os.path.join(EXP_BASE, dataset, "full.csv")))
    if not (len(mu) == len(std) == len(cnf) == n_full_raw):
        raise ValueError(
            f"{dataset} seed {seed}: row-count mismatch "
            f"(full={n_full_raw}, mu={len(mu)}, std={len(std)}, cnf={len(cnf)})"
        )
    # Attach adm1 by row index from the full file (re-loaded WITHOUT dropna so indices align)
    full_raw = pd.read_csv(os.path.join(EXP_BASE, dataset, "full.csv"))
    mu[adm1_col]  = full_raw[adm1_col].values
    std[adm1_col] = full_raw[adm1_col].values
    cnf[adm1_col] = full_raw[adm1_col].values
    return mu, std, cnf

def per_seed_emd(dataset, seed, adm1_col, targets, rng):
    full_clean = load_truth(dataset, adm1_col, targets)
    mu, std, cnf = load_seed_outputs(dataset, seed, RUN_IDX, adm1_col, targets, full_clean)
    rows = []
    adm1s = sorted(full_clean[adm1_col].dropna().astype(str).unique().tolist())
    for adm1 in adm1s:
        full_a = full_clean.loc[full_clean[adm1_col] == adm1]
        mu_a   = mu.loc[mu[adm1_col]   == adm1]
        std_a  = std.loc[std[adm1_col] == adm1]
        cnf_a  = cnf.loc[cnf[adm1_col] == adm1]
        for t in targets:
            t_vals = full_a[t].values
            n_true = int(np.isfinite(t_vals).sum())
            if n_true == 0 or t not in cnf_a.columns or t not in [c.replace('mu_','') for c in mu_a.columns]:
                continue
            mu_col, std_col = f"mu_{t}", f"std_{t}"
            if mu_col not in mu_a.columns or std_col not in std_a.columns:
                continue
            mu_vec  = mu_a[mu_col].values
            std_vec = std_a[std_col].values
            std_safe = np.where(std_vec > 0, std_vec, 1e-9)
            b_vals = rng.normal(mu_vec, std_safe)
            g_vals = cnf_a[t].values
            stack = np.concatenate([_finite_1d(t_vals), _finite_1d(g_vals), _finite_1d(b_vals)])
            if len(stack) == 0: continue
            vr = (float(np.min(stack)), float(np.max(stack)))
            if vr[1] <= vr[0]: continue
            emd_cnf   = emd_1d_match(t_vals, g_vals, n_subsamples=N_SUBSAMPLES, rng=rng,
                                     bins=BINS, value_range=vr)
            emd_bayes = emd_1d_match(t_vals, b_vals, n_subsamples=N_SUBSAMPLES, rng=rng,
                                     bins=BINS, value_range=vr)
            rows.append({
                "dataset": dataset, "seed": seed, "adm1": adm1, "target": t,
                "n_true": n_true,
                "emd_cnf": emd_cnf, "emd_bayes": emd_bayes,
                "imp_cnf_vs_bayes": emd_bayes - emd_cnf,
            })
    return rows

In [ ]:
all_rows = []
for dataset, cfg in DATASETS.items():
    print(f"\n==== {dataset} ====")
    for seed in SEEDS:
        try:
            rng = np.random.default_rng(seed * 9973 + (hash(dataset) % (2**31)))
            seed_rows = per_seed_emd(dataset, seed, cfg["adm1"], cfg["targets"], rng)
            all_rows.extend(seed_rows)
            print(f"  seed {seed}: {len(seed_rows)} (adm1, target) pairs")
        except FileNotFoundError as e:
            print(f"  seed {seed}: missing file {e.filename}")
        except Exception as e:
            print(f"  seed {seed}: skipped ({type(e).__name__}: {e})")

results = pd.DataFrame(all_rows)
print(f"\nTotal rows: {len(results)}")
results.to_csv(os.path.join(OUT_DIR, "cnf_vs_bayes_per_adm1_emd.csv"), index=False)
results.head()

In [ ]:
def cluster_bootstrap(values, pair_ids, n_boot=2000, seed=42, ci=95):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    pair_ids = np.asarray(pair_ids)
    uniq = np.unique(pair_ids)
    idx_map = {p: np.where(pair_ids == p)[0] for p in uniq}
    boots = [np.nanmean(values[np.concatenate([idx_map[p] for p in rng.choice(uniq, size=len(uniq), replace=True)])])
             for _ in range(n_boot)]
    lo_q = (100 - ci) / 2
    return float(np.nanmean(values)), float(np.nanpercentile(boots, lo_q)), float(np.nanpercentile(boots, 100 - lo_q))

summary = []
for (ds, tgt), sub in results.groupby(["dataset", "target"]):
    mean_cnf   = float(np.nanmean(sub["emd_cnf"]))
    mean_bayes = float(np.nanmean(sub["emd_bayes"]))
    mean_imp, lo, hi = cluster_bootstrap(
        sub["imp_cnf_vs_bayes"].values,
        (sub["seed"].astype(str) + "|" + sub["adm1"].astype(str)).values,
    )
    diffs = sub["imp_cnf_vs_bayes"].dropna().values
    try:
        _stat, p = wilcoxon(diffs, alternative="greater")
    except Exception:
        p = np.nan
    summary.append({
        "dataset": ds, "target": tgt, "n_pairs": len(sub),
        "emd_cnf": mean_cnf, "emd_bayes": mean_bayes,
        "abs_improv": mean_imp, "abs_improv_lo": lo, "abs_improv_hi": hi,
        "pct_improv": (100*(mean_bayes-mean_cnf)/mean_bayes) if mean_bayes > 0 else np.nan,
        "wilcoxon_p_one_sided": p,
    })

summary = pd.DataFrame(summary).sort_values(["dataset", "target"]).reset_index(drop=True)
summary.to_csv(os.path.join(OUT_DIR, "cnf_vs_bayes_summary.csv"), index=False)
print(f"Saved {os.path.join(OUT_DIR, 'cnf_vs_bayes_summary.csv')}")
summary.round(4)

In [ ]:
# Headline figure: paired EMD per (dataset, target), cNF vs Bayesian, with diagonal.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=300)
vmax = max(summary["emd_cnf"].max(), summary["emd_bayes"].max()) * 1.1
ax.plot([0, vmax], [0, vmax], color="0.6", linestyle="--", linewidth=0.8, zorder=1)
ax.scatter(summary["emd_bayes"], summary["emd_cnf"], s=24, alpha=0.7, zorder=2)
ax.set_xlim(0, vmax); ax.set_ylim(0, vmax)
ax.set_xlabel("EMD vs truth -- Bayesian regression")
ax.set_ylabel("EMD vs truth -- cNF")
ax.set_title("Distributional accuracy per (dataset, variable)")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
n_below = int((summary["emd_cnf"] < summary["emd_bayes"]).sum())
ax.text(0.02, 0.98, f"{n_below}/{len(summary)} below diagonal (cNF better)",
        transform=ax.transAxes, va="top", ha="left", fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "cnf_vs_bayes_emd_scatter.svg"), bbox_inches="tight")
fig.savefig(os.path.join(OUT_DIR, "cnf_vs_bayes_emd_scatter.pdf"), bbox_inches="tight")
plt.show()